# 📥 Download and Filter NASA GeneLab Omics Datasets

This notebook automates the retrieval and pre‑processing of omics datasets from the NASA GeneLab Open Science Data Repository (OSDR) using the `genelab_utils` package. It supports both incremental and full updates, applies pre‑filters to reduce file size, and writes a manifest of downloaded files.

Author: Peter W. Rose, UC San Diego (pwrose.ucsd@gmail.com)

In [1]:
import os
import pandas as pd
import genelab_utils as gl

pd.set_option("display.max_rows", None)

In [2]:
os.makedirs("../data", exist_ok=True)

In [3]:
MANIFEST_PATH = "../data/manifest.csv" # file to save dataset info

## Incremental vs Full Update
By default, this notebook runs an incremental update. It downloads and preprocesses any new datasets specified in the "technology_types" list below.

If any datasets have been updated, set the "reset" variable to "True" to run a complete update.

The downloaded datasets are saved in the "datasets" directory.

In [4]:
RESET = False # run incremental update
# RESET = True # run a complete update to refresh datasets

In [5]:
info = gl.get_info()
info.head()

,identifier,assay_name,id.sample name,measurement,technology,host_organism,host_strain,material,study.characteristics.material type.term accession number,study.characteristics.organism.term accession number,file.category,file.subcategory,taxonomy
0,OSD-1,OSD-1_transcription-profiling_dna-microarray_A...,Dmel_OR_wo_FLT_infdw-Bbas_Rep1,transcription profiling,DNA microarray,,,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON...,GeneLab Processed Microarray Data Files,Differential gene expression analysis,7227
1,OSD-1,OSD-1_transcription-profiling_dna-microarray_A...,Dmel_OR_wo_GC_infdw-Bbas_Rep3,transcription profiling,DNA microarray,,,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON...,GeneLab Processed Microarray Data Files,Differential gene expression analysis,7227
2,OSD-1,OSD-1_transcription-profiling_dna-microarray_A...,Dmel_OR_wo_GC_infdw-Bbas_Rep3,transcription profiling,DNA microarray,,,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON...,Microarray Data Files,Normalized data,7227
3,OSD-1,OSD-1_transcription-profiling_dna-microarray_A...,Dmel_OR_wo_GC_infdw-Bbas_Rep3,transcription profiling,DNA microarray,,,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON...,Microarray Data Files,Raw array data,7227
4,OSD-1,OSD-1_transcription-profiling_dna-microarray_A...,Dmel_OR_wo_GC_infdw-Bbas_Rep3,transcription profiling,DNA microarray,,,Whole Organism,http://purl.obolibrary.org/obo/NCIT_C13413,http://purl.bioontology.org/ontology/NCBITAXON...,GeneLab Processed Microarray Data Files,Raw Data,7227


## Get a List of GeneLab processed Datasets

In [6]:
dataset_info = gl.get_processed_datasets()

## Filter by Technology Type

In [7]:
technology_types = ["RNA Sequencing (RNA-Seq)", 
                    "DNA microarray", 
                    "Whole Genome Bisulfite Sequencing",
                    "Reduced-Representation Bisulfite Sequencing",
                    "16S",
                    "ITS",
                    "18S",
                    "16S and ITS",
                   ]
dataset_info = gl.filter_by_technology_type(dataset_info, technology_types)

## Filter by Organism

In [8]:
print(f"Available organisms: {dataset_info['taxonomy'].dropna().unique()}")

Available organisms: ['7227' '10090' '6239' '3702' '9606' '' '1423' '13613' '287' '131567'
 '1452' '562' '10116' '7955' '15368' '1781' '3711' '63436' '63433' '4932'
 '148447' '104782' '8090']


In [9]:
taxids = {"9606": "Homo sapiens",
          # -- Rodens -- 
          "10090": "Mus musculus",
          "10116": "Rattus norvegicus",
          # -- Fish --
          "7955": "Danio rerio",
          "8090": "Oryzias latipes",
          # -- Nematoda --
          "6239": "Caenorhabditis elegans",
          # -- Insecta --
          "7227": "Drosophila melanogaster",
          "63436": "Leptopilina heterotoma",
          "63433": "Leptopilina boulardi",
          # -- Bacteria --
          "562": "Escherichia coli",
          "287": "Pseudomonas aeruginosa",
          "1423": "Bacillus subtilis",
          "1781": "Mycobacterium marinum",
          "148447": "Paraburkholderia phymatum",
          # -- Fungi --
          "4932": "Saccharomyces cerevisiae",
          # -- Plants --
          "3711": "Brassica rapa",
          "15368": "Brachypodium distachyon",
          "3702": "Arabidopsis thaliana",
          # -- Other --
          "104782": "Adineta vaga",
          "13613": "Microbiota",
          # -- Host organisms
          "4236": "Lactuca sativa",
         }
          
dataset_info = gl.filter_by_organism(dataset_info, taxids)
dataset_info = gl.filter_by_host_organism(dataset_info, taxids)

In [10]:
print(f"Filtered organisms: {dataset_info['taxonomy'].unique()}")

Filtered organisms: ['7227' '10090' '6239' '3702' '9606' '1423' '13613' '287' '562' '10116'
 '7955' '15368' '1781' '3711' '63436' '63433' '4932' '148447' '104782'
 '8090']


In [11]:
dataset_info = dataset_info[["identifier", "technology", "measurement", "assay_name", "taxonomy", "organism", "material", "host_organism", "host_taxonomy", "host_strain"]].copy()
dataset_info.drop_duplicates(inplace=True)
dataset_info.fillna("", inplace=True)
dataset_info.head(1000)

,identifier,technology,measurement,assay_name,taxonomy,organism,material,host_organism,host_taxonomy,host_strain
203,OSD-1,DNA microarray,transcription profiling,OSD-1_transcription-profiling_dna-microarray_A...,7227,Drosophila melanogaster,Whole Organism,,,
290,OSD-100,RNA Sequencing (RNA-Seq),transcription profiling,OSD-100_transcription-profiling_rna-sequencing...,10090,Mus musculus,left eye,,,
298,OSD-101,RNA Sequencing (RNA-Seq),transcription profiling,OSD-101_transcription-profiling_rna-sequencing...,10090,Mus musculus,Left gastrocnemius,,,
287,OSD-102,RNA Sequencing (RNA-Seq),transcription profiling,OSD-102_transcription-profiling_rna-sequencing...,10090,Mus musculus,Left kidney,,,
286,OSD-103,Whole Genome Bisulfite Sequencing,DNA methylation profiling,OSD-103_dna-methylation-profiling_whole-genome...,10090,Mus musculus,Left quadriceps femoris,,,
294,OSD-103,RNA Sequencing (RNA-Seq),transcription profiling,OSD-103_transcription-profiling_rna-sequencing...,10090,Mus musculus,Left quadriceps femoris,,,
308,OSD-104,RNA Sequencing (RNA-Seq),transcription profiling,OSD-104_transcription-profiling_rna-sequencing...,10090,Mus musculus,Soleus-both sides,,,
304,OSD-105,Whole Genome Bisulfite Sequencing,DNA methylation profiling,OSD-105_dna-methylation-profiling_whole-genome...,10090,Mus musculus,Left tibialis anterior,,,
284,OSD-105,RNA Sequencing (RNA-Seq),transcription profiling,OSD-105_transcription-profiling_rna-sequencing...,10090,Mus musculus,Left tibialis anterior,,,
246,OSD-109,DNA microarray,transcription profiling,OSD-109_transcription-profiling_dna-microarray...,10090,Mus musculus,primary left ventricular cell,,,


## Select Datasets to Download
The map below specifies the technology type and a substring used to identify processed files. Processed files must contain this substring.

In [12]:
file_types = {"DNA microarray": "differential_expression",
              "RNA Sequencing (RNA-Seq)": "differential_expression",
              "Whole Genome Bisulfite Sequencing": "differential_methylation_tiles",
              "Reduced-Representation Bisulfite Sequencing": "differential_methylation_tiles",
              "16S": "differential_abundance",
              "ITS": "differential_abundance",
              "18S": "differential_abundance",
              "16S and ITS": "differential_abundance",
              }

#### Define pre-filters to reduce the file the essential data

In [13]:
def differential_expression_filter(df, threshold=0.1):
    if "ENTREZID" not in df.columns:
        return pd.DataFrame()
        
    filtered_df = df[df['ENTREZID'].notna() & (df['ENTREZID'].astype(str) != '')]
    # Keep only required columns
    filtered_df = filtered_df.filter(regex=r"^(ENTREZID|SYMBOL|GENENAME|Log2fc_|Adj\.p\.value_)")
    adj_pval_cols = [col for col in filtered_df.columns if col.startswith("Adj.p.value_")]
    filtered_df = filtered_df[filtered_df[adj_pval_cols].le(threshold).any(axis=1)]
    # Explode rows with multiple genes
    if "ENTREZID" in filtered_df.columns:
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].astype(str)
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].apply(lambda x:x.split('|'))
        filtered_df = filtered_df.explode('ENTREZID')
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].str.strip()
    filtered_df["differential_analysis_method"] = "DESeq2"
    return filtered_df

In [14]:
def differential_methylation_filter(df, threshold=0.1):
    # add reversed contrast:
    #    meth.diff_(Ground Control)v(Space Flight) -> meth.diff_(Space Flight)v(Ground Control) and reverse sign
    #    qvalue_(Ground Control)v(Space Flight) -> qvalue_(Space Flight)v(Ground Control)
    df = gl.add_reversed_contrast_columns(df)
    
    filtered_df = df[df['ENTREZID'].notna() & (df['ENTREZID'].astype(str) != '')]
    # Keep only required columns
    filtered_df = filtered_df.filter(regex=r"^(ENTREZID|SYMBOL|GENENAME|chr|start|end|dist.to.feature|prom|exon|intron|meth.diff_|qvalue_)")
    qval_cols = [col for col in filtered_df.columns if col.startswith("qvalue_")]
    filtered_df = filtered_df[filtered_df[qval_cols].le(threshold).any(axis=1)]
     # Explode rows with multiple genes
    if "ENTREZID" in filtered_df.columns:
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].astype(str)
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].apply(lambda x:x.split('|'))
        filtered_df = filtered_df.explode('ENTREZID')
        filtered_df["ENTREZID"] = filtered_df["ENTREZID"].str.strip()
    return filtered_df

In [15]:
def differential_abundance_filter(df, threshold=0.1):
    s = df["NCBI_id"]

    mask = (
            s.notna()                               # real NaNs
            & s.astype(str).str.strip().ne("")      # empty/whitespace
            & s.astype(str).str.strip().str.lower().ne("nan")  # literal "nan"
    )        

    filtered_df = df[mask].copy()
    filtered_df["NCBI_id"] = filtered_df["NCBI_id"].astype(int)

    # Keep only required columns
    filtered_df = filtered_df.filter(regex=r"^(best_taxonomy|NCBI_id|Lnfc_|Log2fc_|Q\.value_|Adj\.p\.value_)")
    adj_pval_cols = [col for col in filtered_df.columns if col.startswith("Adj.p.value_")]
    if len(adj_pval_cols) > 0:
        filtered_df = filtered_df[filtered_df[adj_pval_cols].le(threshold).any(axis=1)]
    qval_cols = [col for col in filtered_df.columns if col.startswith("Q.value_")]
    if len(qval_cols) > 0:
        filtered_df = filtered_df[filtered_df[qval_cols].le(threshold).any(axis=1)]

    return filtered_df

In [16]:
filters = {"differential_expression": differential_expression_filter,
           "differential_methylation_tiles": differential_methylation_filter,
           "differential_abundance": differential_abundance_filter,}

In [17]:
manifest = gl.download_data_files(dataset_info, file_types, filters, reset=RESET)
manifest.to_csv(MANIFEST_PATH, index=False)

File already exist: GLDS-1_array_differential_expression.csv
File already exist: GLDS-100_rna_seq_differential_expression.csv
File already exist: GLDS-101_rna_seq_differential_expression_GLbulkRNAseq.csv
File already exist: GLDS-102_rna_seq_differential_expression_GLbulkRNAseq.csv
File already exist: GLDS-103_Gwgbs_differential_methylation_tiles_GLMethylSeq.csv
File already exist: GLDS-103_rna_seq_differential_expression_GLbulkRNAseq.csv
File already exist: GLDS-104_rna_seq_differential_expression_GLbulkRNAseq.csv
File already exist: GLDS-105_Gwgbs_differential_methylation_tiles_GLMethylSeq.csv
File already exist: GLDS-105_rna_seq_differential_expression_GLbulkRNAseq.csv
File already exist: GLDS-109_array_differential_expression_GLmicroarray.csv
File already exist: GLDS-112_array_differential_expression_GLmicroarray.csv
File already exist: GLDS-113_array_differential_expression.csv
File already exist: GLDS-117_array_differential_expression_GLmicroarray.csv
File already exist: GLDS-120_

In [18]:
manifest.head()

,identifier,technology,measurement,assay_name,taxonomy,organism,material,host_organism,host_taxonomy,host_strain,filename,url,differential_analysis_method
203,OSD-1,DNA microarray,transcription profiling,OSD-1_transcription-profiling_dna-microarray_A...,7227,Drosophila melanogaster,Whole Organism,,,,GLDS-1_array_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2
290,OSD-100,RNA Sequencing (RNA-Seq),transcription profiling,OSD-100_transcription-profiling_rna-sequencing...,10090,Mus musculus,left eye,,,,GLDS-100_rna_seq_differential_expression.csv,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2
298,OSD-101,RNA Sequencing (RNA-Seq),transcription profiling,OSD-101_transcription-profiling_rna-sequencing...,10090,Mus musculus,Left gastrocnemius,,,,GLDS-101_rna_seq_differential_expression_GLbul...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2
287,OSD-102,RNA Sequencing (RNA-Seq),transcription profiling,OSD-102_transcription-profiling_rna-sequencing...,10090,Mus musculus,Left kidney,,,,GLDS-102_rna_seq_differential_expression_GLbul...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,DESeq2
286,OSD-103,Whole Genome Bisulfite Sequencing,DNA methylation profiling,OSD-103_dna-methylation-profiling_whole-genome...,10090,Mus musculus,Left quadriceps femoris,,,,GLDS-103_Gwgbs_differential_methylation_tiles_...,https://osdr.nasa.gov/geode-py/ws/studies/OSD-...,methylKit
